# 01 EDA: X Ray dataset

First pass over the nine files in `data/raw`. Goals: know what each table holds, how clean
it is, how much history each company really has, and which raw signals can feed a
group-level monthly health score.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from xray.data import load_all

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

BLUE, ORANGE, AQUA, YELLOW, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#898781"
plt.rcParams.update(
    {
        "figure.figsize": (9, 3.5),
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": MUTED,
        "axes.grid": True,
        "grid.color": "#e8e7e3",
        "grid.linewidth": 0.6,
        "axes.axisbelow": True,
        "axes.titlelocation": "left",
        "axes.titlesize": 11,
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA, YELLOW]),
    }
)

In [ ]:
T = load_all()
groups, companies = T["groups"], T["companies"]
bank, debt, sched = T["banking_products"], T["debt_products"], T["debt_schedule_config"]
tx, inv, bal = T["transactions"], T["invoices"], T["balances"]

## 1. Table overview

In [ ]:
pd.DataFrame(
    {
        "rows": {n: len(d) for n, d in T.items()},
        "cols": {n: d.shape[1] for n, d in T.items()},
        "companies": {
            n: d["company_id"].nunique() if "company_id" in d else np.nan for n, d in T.items()
        },
        "mem_mb": {n: d.memory_usage(deep=True).sum() / 1e6 for n, d in T.items()},
    }
)

In [ ]:
def profile(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "null_pct": df.isna().mean() * 100,
            "nunique": df.nunique(),
        }
    )


for name, df in T.items():
    print(f"\n== {name}")
    print(profile(df).to_string())

There is **no target column** anywhere: no default flag, no health label. Whatever the hidden
test is scored against, we have to build the score from the signals themselves.

## 2. Groups and companies

In [ ]:
size = companies.groupby("group_id").size()
print(size.describe().to_string())
print(
    "groups.n_companies_in_sample matches:",
    (groups.set_index("group_id").n_companies_in_sample == size).all(),
)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
size.value_counts().sort_index().plot.bar(ax=axes[0], color=BLUE, width=0.8)
axes[0].set(title="Companies per group", xlabel="companies", ylabel="groups")
companies.set_index("created_at").resample("QS").size().plot(ax=axes[1], color=BLUE, lw=2)
axes[1].set(title="Companies onboarded per quarter (created_at)", xlabel="")
plt.tight_layout()

In [ ]:
for col in ["country", "currency", "erp"]:
    print(companies[col].value_counts(dropna=False).head(8).to_string(), "\n")

`country` is 82% missing and dirty (`ES`, `ESPAÑA`, `España`): weak feature. `created_at` is
the platform onboarding date, and half of the companies were onboarded **after** the data
window starts. Does their history start at onboarding, or is it backfilled?

In [ ]:
first_tx = tx.groupby("company_id").date.min()
lag = (first_tx - companies.set_index("company_id").created_at).dt.days
print("days from created_at to first transaction:")
print(lag.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_string())

## 3. Banking and debt products

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
bank.type.value_counts().sort_values().plot.barh(ax=axes[0], color=BLUE)
axes[0].set(title="Banking products by type", ylabel="")
debt.type.value_counts().sort_values().plot.barh(ax=axes[1], color=BLUE)
axes[1].set(title="Debt products by type", ylabel="")
plt.tight_layout()

print("companies with any debt product:", debt.company_id.nunique(), "of", len(companies))
print(
    "groups with any debt product:",
    debt.merge(companies[["company_id", "group_id"]], on="company_id").group_id.nunique(),
    "of",
    len(groups),
)

In [ ]:
# Sign convention: debt is stored negative. Positive values are the exception.
print("outstanding sign:", np.sign(debt.outstanding).value_counts().to_dict())
print("granted sign:    ", np.sign(debt.granted).value_counts(dropna=False).to_dict())
debt.groupby("type").agg(
    n=("product_id", "size"),
    granted_null=("granted", lambda s: s.isna().mean()),
    granted_median=("granted", "median"),
    outstanding_median=("outstanding", "median"),
    liquidity_null=("liquidity", lambda s: s.isna().mean()),
)

In [ ]:
# Utilization = outstanding / granted, only where both are negative and granted is known.
d = debt[(debt.granted < 0) & (debt.outstanding <= 0)].assign(
    util=lambda x: x.outstanding / x.granted
)
ax = d[d.util <= 1.5].boxplot(column="util", by="type", figsize=(9, 3.5), color=BLUE)
ax.set(title="Utilization (outstanding / granted) by debt type", xlabel="", ylabel="")
plt.suptitle("")
print("utilization > 1:", (d.util > 1).mean().round(3))

In [ ]:
print(sched.describe(include="all").T.to_string())

Only 87 of 2,239 debt products have a schedule, so installments are not a general feature.
For the rest, debt service has to be read from `debt_repayment` / `interest_charge` transactions.

## 4. Transactions

In [ ]:
tx["month"] = tx.date.dt.to_period("M")
print("date range:", tx.date.min(), "->", tx.date.max())
print("status:", tx.status.value_counts(dropna=False).to_dict())
print("exchange_rate != 1:", (tx.exchange_rate != 1).mean().round(4))

monthly = tx.groupby("month").agg(
    n=("amount", "size"),
    companies=("company_id", "nunique"),
    inflow=("amount", lambda s: s[s > 0].sum()),
)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
monthly.n.plot(ax=axes[0], color=BLUE, lw=2)
axes[0].set(title="Transactions per month", xlabel="", ylim=0)
monthly.companies.plot(ax=axes[1], color=BLUE, lw=2)
axes[1].set(title="Companies with at least one transaction", xlabel="", ylim=0)
plt.tight_layout()
monthly.T

In [ ]:
# How much history does each company and each group really have?
co_months = tx.groupby("company_id").month.nunique()
tx = tx.merge(companies[["company_id", "group_id"]], on="company_id")
gr_months = tx.groupby("group_id").month.nunique()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=False)
co_months.plot.hist(bins=25, ax=axes[0], color=BLUE, rwidth=0.9)
axes[0].set(title="Active months per company", xlabel="months with transactions")
gr_months.plot.hist(bins=25, ax=axes[1], color=BLUE, rwidth=0.9)
axes[1].set(title="Active months per group", xlabel="months with transactions")
plt.tight_layout()
print("companies with >= 12 months:", (co_months >= 12).mean().round(2))
print("groups with >= 12 months:   ", (gr_months >= 12).mean().round(2))
print("groups with >= 24 months:   ", (gr_months >= 24).mean().round(2))

In [ ]:
# Do companies enter late, or do they also go silent before the end?
span = tx.groupby("company_id").month.agg(["min", "max"])
print("first active month:\n", span["min"].value_counts().sort_index().to_string())
print("\nlast active month:\n", span["max"].value_counts().sort_index().tail(8).to_string())

In [ ]:
cat = tx.groupby("category", dropna=False).amount.agg(
    n="size", total="sum", median="median", share_inflow=lambda s: (s > 0).mean()
)
cat.sort_values("n", ascending=False)

Category `-` (uncategorized) is the largest bucket, and `counterparty_id` is 90% empty on
transactions. Amounts are heavy-tailed:

In [ ]:
print(tx.amount.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_string())
ax = np.log10(tx.amount.abs().clip(lower=0.01)).plot.hist(bins=80, color=BLUE)
ax.set(title="log10 |amount| of transactions", xlabel="log10 EUR-ish (mixed currencies)")

In [ ]:
# Transfers between own accounts inflate both inflow and outflow. How big are they?
flows = tx.assign(kind=np.where(tx.amount > 0, "inflow", "outflow")).pivot_table(
    index="category", columns="kind", values="amount", aggfunc="sum"
)
(flows / flows.sum()).sort_values("inflow", ascending=False).head(10)

## 5. Invoices

In [ ]:
print("companies with invoices:", inv.company_id.nunique(), "of", len(companies))
inv = inv.merge(companies[["company_id", "group_id"]], on="company_id")
print("groups with invoices:   ", inv.group_id.nunique(), "of", len(groups))
print("\n", inv.document_type.value_counts().to_string())
print("\n", inv.status.value_counts().to_string())

In [ ]:
# Sign: positive = issued (receivable), negative = received (payable)?
inv["side"] = np.where(inv.amount > 0, "positive", "negative")
print(
    inv.groupby("side").agg(
        n=("amount", "size"), total=("amount", "sum"), counterparties=("counterparty_id", "nunique")
    )
)
pd.crosstab(inv.status, inv.side, normalize="columns").round(3)

In [ ]:
inv["month"] = inv.issuance_date.dt.to_period("M")
in_window = inv[(inv.issuance_date >= "2024-09-01") & (inv.issuance_date <= "2026-09-01")]
print("issued outside the 24 month window:", 1 - len(in_window) / len(inv))
m = in_window.groupby("month").agg(n=("amount", "size"), companies=("company_id", "nunique"))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
m.n.plot(ax=axes[0], color=BLUE, lw=2)
axes[0].set(title="Invoices issued per month", xlabel="", ylim=0)
m.companies.plot(ax=axes[1], color=BLUE, lw=2)
axes[1].set(title="Companies with invoices that month", xlabel="", ylim=0)
plt.tight_layout()

### Payment behaviour
`payment_date` is never null, even for unpaid invoices. What does it hold there?

In [ ]:
unpaid = inv[inv.status.isin(["overdue", "pending"])]
print("unpaid: payment_date == due_date:", (unpaid.payment_date == unpaid.due_date).mean().round(3))
print(
    "unpaid: payment_date in the future (> 2026-09-01):",
    (unpaid.payment_date > "2026-09-01").mean().round(3),
)
print(
    "paid:   pending_amount == 0:", (inv[inv.status == "paid"].pending_amount == 0).mean().round(3)
)

In [ ]:
paid = inv[(inv.status == "paid") & inv.document_type.eq("invoice")].copy()
paid["days_late"] = (paid.payment_date - paid.due_date).dt.days
paid["term"] = (paid.due_date - paid.issuance_date).dt.days
print(
    paid[["days_late", "term"]]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    .to_string()
)

fig, ax = plt.subplots()
for side, color in [("positive", BLUE), ("negative", ORANGE)]:
    paid.loc[paid.side == side, "days_late"].clip(-60, 120).plot.hist(
        bins=90, ax=ax, histtype="step", lw=2, color=color, label=f"{side} amount", density=True
    )
ax.set(title="Days late on paid invoices (clipped to -60..120)", xlabel="payment_date - due_date")
ax.legend(frameon=False)

In [ ]:
# Payment delay over time, split by side: the raw material for DSO / DPO trends.
paid["pay_month"] = paid.payment_date.dt.to_period("M")
w = paid[
    (paid.payment_date >= "2024-09-01")
    & (paid.payment_date <= "2026-08-31")
    & paid.days_late.between(-90, 365)
]
trend = w.pivot_table(index="pay_month", columns="side", values="days_late", aggfunc="mean")
ax = trend.plot(lw=2, color=[ORANGE, BLUE])
ax.set(title="Mean days late by payment month", xlabel="", ylabel="days")
ax.legend(frameon=False)

In [ ]:
# Overdue stock at the snapshot: how old is it, and how concentrated?
ov = inv[inv.status == "overdue"].assign(
    age=lambda x: (pd.Timestamp("2026-09-01") - x.due_date).dt.days
)
print(ov.age.describe(percentiles=[0.05, 0.5, 0.95]).to_string())
by_group = ov.groupby(["group_id", "side"]).pending_amount.sum().unstack(fill_value=0)
print("\ngroups with overdue invoices:", len(by_group))
by_group.describe()

## 6. Balances (snapshot at 1 Sep 2026)

In [ ]:
b = bal.merge(pd.concat([bank, debt])[["product_id", "type"]], on="product_id", how="left")
print("products without a balance row:", len(bank) + len(debt) - len(bal))
b.groupby("type").balance.agg(["size", "median", "min", "max", lambda s: (s < 0).mean()]).rename(
    columns={"<lambda_0>": "share_negative"}
)

In [ ]:
# Can we rebuild a monthly balance history by walking transactions back from the snapshot?
# balance(end of month m) = snapshot - sum(transactions after m)
chk = bank[bank.type == "checking"].product_id
t = tx[tx.product_id.isin(chk)]
flow = t.groupby(["product_id", "month"]).amount.sum().unstack(fill_value=0).sort_index(axis=1)
snap = bal.set_index("product_id").balance.reindex(flow.index)
after = flow.iloc[:, ::-1].cumsum(axis=1).iloc[:, ::-1].shift(-1, axis=1).fillna(0)
hist = (-after).add(snap, axis=0)
print("checking accounts with snapshot and transactions:", snap.notna().sum())
print(
    "share of account-months with negative rebuilt balance:",
    (hist.dropna() < 0).mean().mean().round(3),
)
print("share of accounts negative at snapshot:               ", (snap < 0).mean().round(3))

If the rebuilt balances go negative far more often than the snapshot does, the transaction
feed is incomplete (or amounts are in mixed currencies) and balance history cannot be trusted
as a level. Its trend may still be usable.

## 7. Group monthly panel: first look at trajectories

In [ ]:
# Drop own-account transfers and the partial month of Sep 2026.
core = tx[(tx.category != "transfer") & (tx.month < "2026-09")]
panel = core.groupby(["group_id", "month"]).agg(
    inflow=("amount", lambda s: s[s > 0].sum()),
    outflow=("amount", lambda s: -s[s < 0].sum()),
    n=("amount", "size"),
)
panel["net"] = panel.inflow - panel.outflow
panel["coverage"] = panel.inflow / panel.outflow.replace(0, np.nan)
print(panel.coverage.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_string())

In [ ]:
# Trend of inflows: last 6 full months vs the 6 before, for groups with enough history.
wide = panel.inflow.unstack("month").loc[:, "2025-09":"2026-08"]
full = wide.dropna()
growth = np.log(full.iloc[:, 6:].sum(axis=1) / full.iloc[:, :6].sum(axis=1)).replace(
    [np.inf, -np.inf], np.nan
)
ax = growth.clip(-2, 2).plot.hist(bins=40, color=BLUE, rwidth=0.9)
ax.set(
    title=f"Log inflow growth, last 6 months vs previous 6 ({len(full)} groups)", xlabel="log ratio"
)
print(
    "groups shrinking > 30%:",
    (growth < np.log(0.7)).sum(),
    "| growing > 30%:",
    (growth > np.log(1.3)).sum(),
)

In [ ]:
# Small multiples: six groups from each tail of the growth distribution.
pick = list(growth.nsmallest(3).index) + list(growth.nlargest(3).index)
fig, axes = plt.subplots(2, 3, figsize=(12, 5), sharex=True)
for ax, g in zip(axes.flat, pick, strict=True):
    s = panel.loc[g]
    x = s.index.to_timestamp()
    ax.plot(x, s.inflow, color=BLUE, lw=2, label="inflow")
    ax.plot(x, s.outflow, color=ORANGE, lw=2, label="outflow")
    ax.set_title(f"{g} (log growth {growth[g]:+.2f})")
    ax.tick_params(axis="x", rotation=45)
axes[0, 0].legend(frameon=False)
plt.tight_layout()

The "growing" groups look like step changes, not organic growth. Check whether the group
perimeter (number of companies reporting) changes at the same time.

In [ ]:
n_active = core.groupby(["group_id", "month"]).company_id.nunique().unstack("month")
print(n_active.loc[pick, "2025-03":"2026-08":3].to_string())
changed = (n_active.max(axis=1) > n_active.bfill(axis=1).iloc[:, 0]).mean()
print(f"\ngroups whose number of reporting companies grows inside the window: {changed:.0%}")

## 8. Findings

**Shape of the problem**
- No label in any file. The score is unsupervised or built on proxy outcomes we define
  (overdue build-up, inflow collapse, going silent, debt utilization).
- Panel is unbalanced: 439 companies report in Sep 2024, about 1,200 by 2026. Only 36% of
  groups have all 24 months, 67% have 12 or more. Entry is staggered (waves in Sep 2024,
  Jan 2025, Jan 2026); history is often backfilled before `created_at`, so use the first
  transaction, not `created_at`, as the start of observation.
- Group perimeter changes: new companies get connected mid-window, which looks like growth in
  group totals. Score on a fixed perimeter or per company, then aggregate.
- A few companies go silent before Aug 2026 (about 50 stop by Jun 2026): candidate distress proxy.
- Sep 2026 is a partial month (one day). Drop it from monthly features.

**Transactions** (2.56M rows, all 1,286 companies)
- 25% are category `-`, carrying about 40% of the money both ways. `transfer` is another
  15 to 22%: exclude it from operating flows.
- Useful named flows: `collection`, `payment`, `salary`, `social_security`, `tax`,
  `debt_repayment`, `interest_charge`, `fee`, `collection_refund`.
- `counterparty_id` is 90% empty here, so customer concentration has to come from invoices.
- Amounts are in account currency (4% have fx != 1) and extremely heavy-tailed (single
  movements of 3e9). Use ratios, logs and robust statistics, never raw sums across groups.

**Invoices** (898K rows, only 785 companies and 167 of 250 groups)
- Any invoice feature needs a missing-indicator: a third of groups have none.
- Sign: positive looks like receivables, negative like payables. Confirm with the organizers.
- `payment_date` is never null: for unpaid invoices it equals `due_date` (expected date), so it
  is only a real payment date when `status == "paid"`.
- `status` and `pending_amount` are the state at extraction, i.e. month-24 information.
  For month t, rebuild "open and past due at t" from issuance, due and paid dates only.
- 21% of invoices are overdue, median age 185 days: the overdue stock is large and old.
- Date garbage exists (terms of 1.8M days, year-1 style dates): clip to a sane range.
- Invoice volume grows with onboarding, so compare rates, not counts.

**Products and balances**
- Debt is stored negative; 145 products have positive outstanding and 169 lack `granted`.
  378 companies have debt. Only 87 products have a schedule.
- Credit lines, confirming and factoring report `liquidity`, so utilization is computable for them.
- Balances contain sentinels (-999,999,999 and 99,999,990,850): filter before use.
- Rebuilding balance history from the snapshot gives negative checking balances in 10% of
  account-months vs 2% at the snapshot: the feed is not complete enough to trust levels.

**Candidate drivers for the score**
1. Operating coverage: inflow / outflow excluding transfers, 3 month rolling.
2. Inflow trend and volatility, per company, on a fixed perimeter.
3. Collection delay (DSO-like) and its trend; share of receivables past due at t.
4. Own payment delay (DPO-like): paying suppliers later is an early stress signal.
5. Fixed-cost burden: salary + social security + tax over inflow; missed payroll months.
6. Debt service over inflow; fees, interest charges and `collection_refund` (returned receipts) rates.
7. Activity: transaction count trend and silence.